# Train drilling advisory models: rotation → future ROP

Цель: обучить модель, которая подбирает `pressure_axis` и `pressure_rotation` для повышения будущего ROP.

Цепочка:

```text
candidate p_ax, p_rot + current telemetry/history/energy_type
    → predicted future rotation
    → predicted future average speed
```

Вход: `united_rock_energy_segment_quantile.csv`, полученный из notebook с `energy_type_segment_quantile`.

Выход: папка `drilling_advisory_artifacts/` с моделями, конфигами и offline-рекомендациями.

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import json
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score

try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    from sklearn.metrics import mean_squared_error
    def root_mean_squared_error(y_true, y_pred):
        return mean_squared_error(y_true, y_pred, squared=False)

try:
    import lightgbm as lgb
    HAS_LIGHTGBM = True
except Exception:
    HAS_LIGHTGBM = False

import joblib

RANDOM_STATE = 42
EPS = 1e-6

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

## 1. Загрузка датасета

In [2]:
CANDIDATE_PATHS = [
    "united_rock_energy_segment_quantile.csv",
    "../united_rock_energy_segment_quantile.csv",
    "notebooks/united_rock_energy_segment_quantile.csv",
    "../notebooks/united_rock_energy_segment_quantile.csv",
]

DATA_PATH = next((p for p in CANDIDATE_PATHS if Path(p).exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Не найден united_rock_energy_segment_quantile.csv. Сначала запусти rock_energy_segment_quantile_surfaces_only.ipynb")

df = pd.read_csv(DATA_PATH).drop(columns=["Unnamed: 0"], errors="ignore")

required_cols = [
    "processing_time", "well_id", "pressure_axis", "pressure_rotation",
    "rotation", "speed", "hardness_score_smooth", "rock_energy_type_final"
]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Не хватает колонок: {missing}")

df["processing_time"] = pd.to_datetime(df["processing_time"])
df = df.sort_values(["well_id", "processing_time"]).reset_index(drop=True)
df["rock_energy_type_final"] = df["rock_energy_type_final"].fillna("unknown").astype(str)

print("Loaded:", DATA_PATH)
print("Shape:", df.shape)
display(df[required_cols].head())
display(df[["pressure_axis", "pressure_rotation", "rotation", "speed", "hardness_score_smooth"]].describe(percentiles=[.01,.05,.5,.95,.99]))

Loaded: united_rock_energy_segment_quantile.csv
Shape: (415049, 118)


,processing_time,well_id,pressure_axis,pressure_rotation,rotation,speed,hardness_score_smooth,rock_energy_type_final
0,2025-08-24 10:09:50.980,19601,804,4551,74.256,0.002755,NaN,unknown
1,2025-08-24 10:10:00.260,19601,763,3782,73.812,0.003030,NaN,unknown
2,2025-08-24 10:10:05.199,19601,879,4407,73.512,0.006060,NaN,unknown
3,2025-08-24 10:10:14.610,19601,721,3705,73.962,0.002755,NaN,unknown
4,2025-08-24 10:10:34.197,19601,859,3883,74.256,0.001515,NaN,unknown


,pressure_axis,pressure_rotation,rotation,speed,hardness_score_smooth
count,415049.000000,415049.000000,415049.000000,415049.000000,3.826350e+05
mean,17473.667273,14134.146325,103.945315,0.013116,4.259450e-15
std,4682.982816,3243.523440,13.370361,0.006615,1.898597e+00
min,317.000000,784.000000,50.010000,0.001002,-1.023758e+01
1%,3713.000000,6271.000000,64.980000,0.002755,-5.313689e+00
5%,6645.000000,8246.000000,81.750000,0.005050,-3.437390e+00
50%,18861.000000,14637.000000,103.158000,0.012120,2.413973e-01
95%,22343.000000,18758.600000,138.474000,0.024240,2.752807e+00
99%,23626.000000,20881.000000,139.020000,0.030300,3.826881e+00
max,24872.000000,26318.000000,139.578000,0.038957,6.586316e+00


## 2. Feature engineering

In [3]:
df["total_pressure"] = df["pressure_axis"] + df["pressure_rotation"]
df["pressure_balance"] = df["pressure_axis"] / (df["pressure_axis"] + df["pressure_rotation"] + EPS)
df["axis_over_rot_pressure"] = df["pressure_axis"] / (df["pressure_rotation"] + EPS)
df["rot_pressure_over_axis"] = df["pressure_rotation"] / (df["pressure_axis"] + EPS)
df["rotation_efficiency"] = df["rotation"] / (df["pressure_rotation"] + EPS)
df["axis_x_rotation"] = df["pressure_axis"] * df["rotation"]
df["rot_pressure_x_rotation"] = df["pressure_rotation"] * df["rotation"]
df["energy_input_proxy"] = df["pressure_axis"] + df["pressure_rotation"] * df["rotation"]
df["log_energy_input_proxy"] = np.log1p(df["energy_input_proxy"])

df["dt"] = df.groupby("well_id")["processing_time"].diff().dt.total_seconds()
df["dt"] = df["dt"].fillna(df["dt"].median())

history_cols = ["pressure_axis", "pressure_rotation", "rotation", "speed", "hardness_score_smooth", "energy_input_proxy", "pressure_balance"]
for col in history_cols:
    for lag in [1, 3, 6, 12]:
        df[f"{col}_lag{lag}"] = df.groupby("well_id")[col].shift(lag)
    shifted = df.groupby("well_id")[col].shift(1)
    for w in [6, 12, 30, 60]:
        min_p = max(2, w // 3)
        df[f"{col}_roll_mean_{w}"] = shifted.groupby(df["well_id"]).rolling(w, min_periods=min_p).mean().reset_index(level=0, drop=True)
        df[f"{col}_roll_std_{w}"] = shifted.groupby(df["well_id"]).rolling(w, min_periods=min_p).std().reset_index(level=0, drop=True)

for col in ["pressure_axis", "pressure_rotation", "rotation", "speed", "hardness_score_smooth"]:
    prev = df.groupby("well_id")[col].shift(1)
    df[f"{col}_diff1"] = df[col] - prev
    df[f"{col}_rel_diff1"] = df[f"{col}_diff1"] / (prev.abs() + EPS)

print("features created")

features created


## 3. Future targets

In [4]:
ROTATION_HORIZON = 30
SPEED_HORIZON = 60

df["future_rotation"] = df.groupby("well_id")["rotation"].shift(-ROTATION_HORIZON)

# Средняя скорость на следующих SPEED_HORIZON точках.
def future_avg(s, horizon):
    return s.shift(-1).rolling(horizon, min_periods=max(5, horizon // 3)).mean().shift(-(horizon - 1))

df["future_avg_speed"] = df.groupby("well_id")["speed"].transform(lambda s: future_avg(s, SPEED_HORIZON))

display(df[["rotation", "future_rotation", "speed", "future_avg_speed"]].describe(percentiles=[.01,.05,.5,.95,.99]))

,rotation,future_rotation,speed,future_avg_speed
count,415049.000000,363869.000000,415049.000000,314464.000000
mean,103.945315,104.523818,0.013116,0.012866
std,13.370361,12.071824,0.006615,0.004162
min,50.010000,50.094000,0.001002,0.002772
1%,64.980000,73.140000,0.002755,0.005434
5%,81.750000,88.548000,0.005050,0.006938
50%,103.158000,103.110000,0.012120,0.012409
95%,138.474000,138.450000,0.024240,0.020591
99%,139.020000,139.014000,0.030300,0.024206
max,139.578000,139.578000,0.038957,0.031033


## 4. Features

In [5]:
base_numeric_features = [
    "pressure_axis", "pressure_rotation", "total_pressure", "pressure_balance", "axis_over_rot_pressure", "rot_pressure_over_axis",
    "rotation", "speed", "hardness_score_smooth", "dt",
    "rotation_efficiency", "axis_x_rotation", "rot_pressure_x_rotation", "energy_input_proxy", "log_energy_input_proxy",
    "pressure_axis_lag1", "pressure_axis_lag3", "pressure_axis_lag6",
    "pressure_rotation_lag1", "pressure_rotation_lag3", "pressure_rotation_lag6",
    "rotation_lag1", "rotation_lag3", "rotation_lag6",
    "speed_lag1", "speed_lag3", "speed_lag6",
    "hardness_score_smooth_lag1", "hardness_score_smooth_lag3", "hardness_score_smooth_lag6",
    "pressure_axis_roll_mean_12", "pressure_axis_roll_std_12",
    "pressure_rotation_roll_mean_12", "pressure_rotation_roll_std_12",
    "rotation_roll_mean_12", "rotation_roll_std_12",
    "speed_roll_mean_12", "speed_roll_std_12",
    "hardness_score_smooth_roll_mean_12", "hardness_score_smooth_roll_std_12",
    "pressure_axis_diff1", "pressure_rotation_diff1", "rotation_diff1", "speed_diff1", "hardness_score_smooth_diff1",
]

categorical_features = ["rock_energy_type_final"]
speed_extra_features = ["candidate_future_rotation"]
speed_numeric_features = base_numeric_features + speed_extra_features

target_cols = ["future_rotation", "future_avg_speed"]
model_df = df.dropna(subset=base_numeric_features + categorical_features + target_cols).copy()

print("Model df:", model_df.shape)
display(model_df[base_numeric_features + target_cols].describe(percentiles=[.01,.05,.5,.95,.99]))

Model df: (272581, 190)


,pressure_axis,pressure_rotation,total_pressure,pressure_balance,axis_over_rot_pressure,rot_pressure_over_axis,rotation,speed,hardness_score_smooth,dt,rotation_efficiency,axis_x_rotation,rot_pressure_x_rotation,energy_input_proxy,log_energy_input_proxy,pressure_axis_lag1,pressure_axis_lag3,pressure_axis_lag6,pressure_rotation_lag1,pressure_rotation_lag3,pressure_rotation_lag6,rotation_lag1,rotation_lag3,rotation_lag6,speed_lag1,speed_lag3,speed_lag6,hardness_score_smooth_lag1,hardness_score_smooth_lag3,hardness_score_smooth_lag6,pressure_axis_roll_mean_12,pressure_axis_roll_std_12,pressure_rotation_roll_mean_12,pressure_rotation_roll_std_12,rotation_roll_mean_12,rotation_roll_std_12,speed_roll_mean_12,speed_roll_std_12,hardness_score_smooth_roll_mean_12,hardness_score_smooth_roll_std_12,pressure_axis_diff1,pressure_rotation_diff1,rotation_diff1,speed_diff1,hardness_score_smooth_diff1,future_rotation,future_avg_speed
count,272581.000000,272581.000000,272581.000000,272581.000000,272581.000000,272581.000000,272581.000000,272581.000000,272581.000000,272581.000000,272581.000000,2.725810e+05,2.725810e+05,2.725810e+05,272581.000000,272581.000000,272581.000000,272581.000000,272581.000000,272581.000000,272581.000000,272581.000000,272581.000000,272581.000000,272581.000000,272581.000000,272581.000000,272581.000000,272581.000000,272581.000000,272581.000000,272581.000000,272581.000000,272581.000000,272581.000000,272581.000000,272581.000000,272581.000000,272581.000000,272581.000000,272581.000000,272581.000000,272581.000000,272581.000000,272581.000000,272581.000000,272581.000000
mean,18113.067044,14304.352853,32417.419897,0.556707,1.274389,0.811639,104.117752,0.012645,0.112669,7.191415,0.007617,1.892122e+06,1.493884e+06,1.511997e+06,14.196460,18074.518334,17994.138128,17863.884515,14280.580580,14231.032185,14152.006365,104.099304,104.061395,104.006977,0.012638,0.012625,0.012606,0.108201,0.099018,0.084614,17833.098027,527.790389,14135.972220,1152.638220,103.983376,4.850316,0.012601,0.003992,0.082665,0.133622,38.548710,23.772273,0.018447,0.000007,0.004467,104.382605,0.012717
std,3754.237783,2906.491260,6264.663613,0.042710,0.201663,0.273779,12.133865,0.006263,1.816581,107.298349,0.002101,4.641648e+05,3.570651e+05,3.596969e+05,0.267857,3788.226556,3859.158159,3975.021318,2918.185461,2942.513999,2979.801906,12.162312,12.223417,12.337283,0.006267,0.006275,0.006289,1.820429,1.828595,1.842452,3854.701009,1020.995240,2712.453214,627.808433,9.217907,7.116189,0.004809,0.001419,1.835399,0.125457,1034.669875,1420.170516,12.659322,0.005966,0.089696,11.500681,0.004032
min,555.000000,1101.000000,2762.000000,0.021882,0.022372,0.071221,50.010000,0.001002,-9.372558,0.092000,0.002104,4.658381e+04,7.073044e+04,7.294844e+04,11.197522,555.000000,555.000000,555.000000,1101.000000,1101.000000,1101.000000,50.010000,50.010000,50.010000,0.001002,0.001002,0.001002,-9.752546,-9.989748,-10.237577,1334.333333,1.831955,3614.333333,71.951162,52.908000,0.078126,0.001676,0.000292,-9.985086,0.000000,-22422.000000,-21094.000000,-88.104000,-0.035202,-1.292160,50.394000,0.002772
1%,5686.800000,7391.800000,13643.400000,0.400971,0.669368,0.584544,71.292000,0.002755,-4.993285,0.228000,0.004872,5.264451e+05,6.055752e+05,6.154206e+05,13.330063,5603.800000,5456.800000,5132.000000,7358.000000,7289.000000,7176.000000,71.214000,70.848000,70.254000,0.002755,0.002755,0.002755,-5.014609,-5.057845,-5.129683,5845.800000,4.056420,7617.366667,351.694754,72.178800,0.349075,0.004203,0.001280,-5.109063,0.011345,-1571.200000,-3526.200000,-36.630000,-0.013130,-0.252225,75.366000,0.005441
5%,10130.000000,9071.000000,19869.000000,0.486780,0.948480,0.638913,85.752000,0.005050,-3.147536,0.460000,0.005464,9.909684e+05,8.712969e+05,8.835475e+05,13.691702,9959.000000,9759.000000,9326.000000,9037.000000,8959.000000,8840.000000,85.518000,85.074000,84.486000,0.005050,0.005050,0.005050,-3.159980,-3.180044,-3.220053,9613.083333,5.736645,9148.833333,495.276254,86.659000,0.476275,0.005892,0.002071,-3.208110,0.0216

## 5. Split по unseen wells

In [6]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(model_df, groups=model_df["well_id"]))
train_df = model_df.iloc[train_idx].copy()
test_df = model_df.iloc[test_idx].copy()

print("Train:", train_df.shape, "wells:", train_df["well_id"].nunique())
print("Test:", test_df.shape, "wells:", test_df["well_id"].nunique())

Train: (204651, 190) wells: 1241
Test: (67930, 190) wells: 414


## 6. Обучение rotation_model

In [7]:
def make_regressor():
    if HAS_LIGHTGBM:
        return lgb.LGBMRegressor(
            n_estimators=700, learning_rate=0.03, num_leaves=63,
            subsample=0.9, colsample_bytree=0.9,
            random_state=RANDOM_STATE, objective="regression", verbosity=-1,
        )
    return HistGradientBoostingRegressor(
        max_iter=500, learning_rate=0.04, max_leaf_nodes=63,
        l2_regularization=0.01, random_state=RANDOM_STATE,
    )

def make_preprocessor(numeric_features, categorical_features):
    return ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
            ("num", "passthrough", numeric_features),
        ],
        remainder="drop",
    )

rotation_model = Pipeline(steps=[
    ("preprocess", make_preprocessor(base_numeric_features, categorical_features)),
    ("model", make_regressor()),
])

rotation_model.fit(train_df[base_numeric_features + categorical_features], train_df["future_rotation"])
test_df["pred_future_rotation"] = rotation_model.predict(test_df[base_numeric_features + categorical_features])

rotation_metrics = {
    "MAE": mean_absolute_error(test_df["future_rotation"], test_df["pred_future_rotation"]),
    "RMSE": root_mean_squared_error(test_df["future_rotation"], test_df["pred_future_rotation"]),
    "R2": r2_score(test_df["future_rotation"], test_df["pred_future_rotation"]),
}
print(rotation_metrics)

{'MAE': 5.239960753127747, 'RMSE': 9.031000789046164, 'R2': 0.41413846848913194}


## 7. Обучение future_speed_model

In [8]:
train_speed_df = train_df.copy()
train_speed_df["candidate_future_rotation"] = train_speed_df["future_rotation"]

test_speed_oracle_df = test_df.copy()
test_speed_oracle_df["candidate_future_rotation"] = test_speed_oracle_df["future_rotation"]

test_speed_chained_df = test_df.copy()
test_speed_chained_df["candidate_future_rotation"] = test_speed_chained_df["pred_future_rotation"]

speed_model = Pipeline(steps=[
    ("preprocess", make_preprocessor(speed_numeric_features, categorical_features)),
    ("model", make_regressor()),
])

speed_model.fit(train_speed_df[speed_numeric_features + categorical_features], train_speed_df["future_avg_speed"])

test_df["pred_future_speed_oracle_rotation"] = speed_model.predict(test_speed_oracle_df[speed_numeric_features + categorical_features])
test_df["pred_future_speed_chained"] = speed_model.predict(test_speed_chained_df[speed_numeric_features + categorical_features])

speed_metrics_oracle = {
    "MAE": mean_absolute_error(test_df["future_avg_speed"], test_df["pred_future_speed_oracle_rotation"]),
    "RMSE": root_mean_squared_error(test_df["future_avg_speed"], test_df["pred_future_speed_oracle_rotation"]),
    "R2": r2_score(test_df["future_avg_speed"], test_df["pred_future_speed_oracle_rotation"]),
}
speed_metrics_chained = {
    "MAE": mean_absolute_error(test_df["future_avg_speed"], test_df["pred_future_speed_chained"]),
    "RMSE": root_mean_squared_error(test_df["future_avg_speed"], test_df["pred_future_speed_chained"]),
    "R2": r2_score(test_df["future_avg_speed"], test_df["pred_future_speed_chained"]),
}
print("speed oracle:", speed_metrics_oracle)
print("speed chained:", speed_metrics_chained)

speed oracle: {'MAE': 0.0019367113755206127, 'RMSE': 0.002526891754391154, 'R2': 0.5962466668984445}
speed chained: {'MAE': 0.002034296913349755, 'RMSE': 0.0026567679086939256, 'R2': 0.5536761675510022}


## 8. Baselines

In [9]:
test_df["baseline_current_speed"] = test_df["speed"]
test_df["baseline_speed_roll_mean_12"] = test_df["speed_roll_mean_12"].fillna(test_df["speed"])

baseline_compare = pd.DataFrame([
    {"model": "current_speed", "MAE": mean_absolute_error(test_df["future_avg_speed"], test_df["baseline_current_speed"]), "RMSE": root_mean_squared_error(test_df["future_avg_speed"], test_df["baseline_current_speed"]), "R2": r2_score(test_df["future_avg_speed"], test_df["baseline_current_speed"])},
    {"model": "speed_roll_mean_12", "MAE": mean_absolute_error(test_df["future_avg_speed"], test_df["baseline_speed_roll_mean_12"]), "RMSE": root_mean_squared_error(test_df["future_avg_speed"], test_df["baseline_speed_roll_mean_12"]), "R2": r2_score(test_df["future_avg_speed"], test_df["baseline_speed_roll_mean_12"])},
    {"model": "rotation_to_speed_chained", "MAE": speed_metrics_chained["MAE"], "RMSE": speed_metrics_chained["RMSE"], "R2": speed_metrics_chained["R2"]},
]).sort_values("MAE")
display(baseline_compare)

,model,MAE,RMSE,R2
2,rotation_to_speed_chained,0.002034,0.002657,0.553676
1,speed_roll_mean_12,0.002699,0.003602,0.179669
0,current_speed,0.004019,0.005132,-0.665422


## 9. Surface ranges по energy type

In [10]:
energy_types = ["soft_low_energy", "medium_low_energy", "medium_high_energy", "hard_high_energy"]
surface_ranges = {}
for et in energy_types:
    part = train_df[train_df["rock_energy_type_final"] == et]
    if len(part) == 0:
        continue
    surface_ranges[et] = {
        "pressure_axis_q05": float(part["pressure_axis"].quantile(0.05)),
        "pressure_axis_q95": float(part["pressure_axis"].quantile(0.95)),
        "pressure_rotation_q05": float(part["pressure_rotation"].quantile(0.05)),
        "pressure_rotation_q95": float(part["pressure_rotation"].quantile(0.95)),
        "rows": int(len(part)),
    }
display(pd.DataFrame(surface_ranges).T)

,pressure_axis_q05,pressure_axis_q95,pressure_rotation_q05,pressure_rotation_q95,rows
soft_low_energy,8035.00,21790.0,8558.3,18778.70,44947.0
medium_low_energy,10598.00,22310.0,9315.0,18655.95,51542.0
medium_high_energy,12127.00,22398.0,9460.0,18450.00,53396.0
hard_high_energy,11825.25,22434.0,9273.0,18333.00,54766.0


## 10. Constrained optimizer

In [11]:
def recompute_candidate_features(grid):
    grid = grid.copy()
    grid["total_pressure"] = grid["pressure_axis"] + grid["pressure_rotation"]
    grid["pressure_balance"] = grid["pressure_axis"] / (grid["pressure_axis"] + grid["pressure_rotation"] + EPS)
    grid["axis_over_rot_pressure"] = grid["pressure_axis"] / (grid["pressure_rotation"] + EPS)
    grid["rot_pressure_over_axis"] = grid["pressure_rotation"] / (grid["pressure_axis"] + EPS)
    grid["rotation_efficiency"] = grid["rotation"] / (grid["pressure_rotation"] + EPS)
    grid["axis_x_rotation"] = grid["pressure_axis"] * grid["rotation"]
    grid["rot_pressure_x_rotation"] = grid["pressure_rotation"] * grid["rotation"]
    grid["energy_input_proxy"] = grid["pressure_axis"] + grid["pressure_rotation"] * grid["rotation"]
    grid["log_energy_input_proxy"] = np.log1p(grid["energy_input_proxy"])
    return grid

def build_candidate_grid(row, grid_size=25, max_delta_frac=0.08):
    et = row["rock_energy_type_final"]
    if et in surface_ranges:
        r = surface_ranges[et]
        p_ax_low, p_ax_high = r["pressure_axis_q05"], r["pressure_axis_q95"]
        p_rot_low, p_rot_high = r["pressure_rotation_q05"], r["pressure_rotation_q95"]
    else:
        p_ax_low, p_ax_high = train_df["pressure_axis"].quantile([.05,.95])
        p_rot_low, p_rot_high = train_df["pressure_rotation"].quantile([.05,.95])

    cur_ax, cur_rotp = row["pressure_axis"], row["pressure_rotation"]
    p_ax_min = max(p_ax_low, cur_ax * (1 - max_delta_frac))
    p_ax_max = min(p_ax_high, cur_ax * (1 + max_delta_frac))
    p_rot_min = max(p_rot_low, cur_rotp * (1 - max_delta_frac))
    p_rot_max = min(p_rot_high, cur_rotp * (1 + max_delta_frac))
    if p_ax_min >= p_ax_max:
        p_ax_min, p_ax_max = cur_ax * (1 - max_delta_frac), cur_ax * (1 + max_delta_frac)
    if p_rot_min >= p_rot_max:
        p_rot_min, p_rot_max = cur_rotp * (1 - max_delta_frac), cur_rotp * (1 + max_delta_frac)

    p_ax_grid = np.linspace(p_ax_min, p_ax_max, grid_size)
    p_rot_grid = np.linspace(p_rot_min, p_rot_max, grid_size)
    PA, PR = np.meshgrid(p_ax_grid, p_rot_grid)
    grid = pd.DataFrame({"pressure_axis": PA.ravel(), "pressure_rotation": PR.ravel()})

    derived = {"pressure_axis", "pressure_rotation", "total_pressure", "pressure_balance", "axis_over_rot_pressure", "rot_pressure_over_axis", "rotation_efficiency", "axis_x_rotation", "rot_pressure_x_rotation", "energy_input_proxy", "log_energy_input_proxy"}
    for col in base_numeric_features:
        if col not in derived:
            grid[col] = row[col]
    for col in categorical_features:
        grid[col] = row[col]
    grid = recompute_candidate_features(grid)
    return grid, PA, PR

def recommend_for_row(row, grid_size=25, max_delta_frac=0.08):
    grid, PA, PR = build_candidate_grid(row, grid_size=grid_size, max_delta_frac=max_delta_frac)
    grid["candidate_future_rotation"] = rotation_model.predict(grid[base_numeric_features + categorical_features])
    grid["pred_future_speed"] = speed_model.predict(grid[speed_numeric_features + categorical_features])
    best_idx = int(grid["pred_future_speed"].values.argmax())
    best = grid.iloc[best_idx]

    current_grid = pd.DataFrame([row[base_numeric_features + categorical_features].to_dict()])
    current_grid["candidate_future_rotation"] = rotation_model.predict(current_grid[base_numeric_features + categorical_features])
    current_pred_speed = float(speed_model.predict(current_grid[speed_numeric_features + categorical_features])[0])

    return {
        "recommended_pressure_axis": float(best["pressure_axis"]),
        "recommended_pressure_rotation": float(best["pressure_rotation"]),
        "predicted_future_rotation": float(best["candidate_future_rotation"]),
        "predicted_future_speed": float(best["pred_future_speed"]),
        "current_predicted_future_speed": current_pred_speed,
        "predicted_uplift_pct": 100.0 * (float(best["pred_future_speed"]) / (current_pred_speed + EPS) - 1.0),
        "grid": grid,
        "PA": PA,
        "PR": PR,
        "Z": grid["pred_future_speed"].values.reshape(PA.shape),
    }
print("optimizer ready")

optimizer ready


## 11. Offline replay evaluation

In [12]:
EVAL_N = min(2000, len(test_df))
eval_points = test_df.sample(EVAL_N, random_state=RANDOM_STATE).copy()
recs = []
for _, row in eval_points.iterrows():
    rec = recommend_for_row(row, grid_size=21, max_delta_frac=0.08)
    recs.append({
        "well_id": row["well_id"],
        "processing_time": row["processing_time"],
        "rock_energy_type_final": row["rock_energy_type_final"],
        "operator_pressure_axis": row["pressure_axis"],
        "operator_pressure_rotation": row["pressure_rotation"],
        "recommended_pressure_axis": rec["recommended_pressure_axis"],
        "recommended_pressure_rotation": rec["recommended_pressure_rotation"],
        "current_speed": row["speed"],
        "future_avg_speed_actual": row["future_avg_speed"],
        "current_predicted_future_speed": rec["current_predicted_future_speed"],
        "recommended_predicted_future_speed": rec["predicted_future_speed"],
        "predicted_uplift_pct": rec["predicted_uplift_pct"],
        "delta_pressure_axis_pct": 100.0 * (rec["recommended_pressure_axis"] / (row["pressure_axis"] + EPS) - 1.0),
        "delta_pressure_rotation_pct": 100.0 * (rec["recommended_pressure_rotation"] / (row["pressure_rotation"] + EPS) - 1.0),
    })
rec_df = pd.DataFrame(recs)
display(rec_df.head())
uplift_summary = rec_df["predicted_uplift_pct"].describe(percentiles=[.05,.25,.5,.75,.95])
display(uplift_summary)
by_energy = rec_df.groupby("rock_energy_type_final").agg(
    rows=("predicted_uplift_pct", "size"),
    median_uplift_pct=("predicted_uplift_pct", "median"),
    mean_uplift_pct=("predicted_uplift_pct", "mean"),
    p05_uplift_pct=("predicted_uplift_pct", lambda s: s.quantile(.05)),
    p95_uplift_pct=("predicted_uplift_pct", lambda s: s.quantile(.95)),
    median_delta_axis_pct=("delta_pressure_axis_pct", "median"),
    median_delta_rot_pct=("delta_pressure_rotation_pct", "median"),
).sort_index()
display(by_energy)

,well_id,processing_time,rock_energy_type_final,operator_pressure_axis,operator_pressure_rotation,recommended_pressure_axis,recommended_pressure_rotation,current_speed,future_avg_speed_actual,current_predicted_future_speed,recommended_predicted_future_speed,predicted_uplift_pct,delta_pressure_axis_pct,delta_pressure_rotation_pct
0,20784,2025-09-02 03:33:04.918,soft_low_energy,10260,9795,10998.720,9481.560,0.02020,0.014969,0.017986,0.018292,1.693036,7.200000,-3.2
1,21424,2025-09-08 02:35:41.239,hard_high_energy,22144,15555,22330.924,16799.400,0.00606,0.011379,0.009771,0.009844,0.741375,0.844129,8.0
2,29328,2025-10-31 05:21:44.297,medium_low_energy,20421,15331,21891.312,16312.184,0.01212,0.014213,0.014552,0.014728,1.201455,7.200000,6.4
3,27941,2025-10-23 04:44:03.254,medium_low_energy,17818,12746,17390.368,13765.680,0.00606,0.012823,0.013416,0.013647,1.713111,-2.400000,8.0
4,23871,2025-09-25 21:58:15.182,medium_low_energy,14127,13185,15257.160,12446.640,0.01010,0.013786,0.012897,0.013125,1.756478,8.000000,-5.6


count    2000.000000
mean        2.262616
std         2.234534
min        -7.502753
5%          0.134146
25%         0.931240
50%         1.735043
75%         3.041668
95%         6.401802
max        22.787921
Name: predicted_uplift_pct, dtype: float64

,rows,median_uplift_pct,mean_uplift_pct,p05_uplift_pct,p95_uplift_pct,median_delta_axis_pct,median_delta_rot_pct
rock_energy_type_final,,,,,,,
hard_high_energy,495,2.508652,3.097676,0.228723,8.115647,2.078539,6.400000
medium_high_energy,518,1.808681,2.239322,0.239724,5.306964,1.413291,5.601876
medium_low_energy,519,1.417070,1.757585,0.110645,4.809310,3.200000,4.643020
soft_low_energy,468,1.570242,1.965230,0.005671,5.671045,0.500882,-0.800000


## 12. Boundary check

In [13]:
boundary_tol = 0.95 * 8.0
rec_df["axis_near_boundary"] = rec_df["delta_pressure_axis_pct"].abs() >= boundary_tol
rec_df["rot_near_boundary"] = rec_df["delta_pressure_rotation_pct"].abs() >= boundary_tol
rec_df["any_boundary"] = rec_df["axis_near_boundary"] | rec_df["rot_near_boundary"]
boundary_report = {
    "axis_near_boundary_ratio": float(rec_df["axis_near_boundary"].mean()),
    "rot_near_boundary_ratio": float(rec_df["rot_near_boundary"].mean()),
    "any_boundary_ratio": float(rec_df["any_boundary"].mean()),
}
print(boundary_report)
display(rec_df[["predicted_uplift_pct", "delta_pressure_axis_pct", "delta_pressure_rotation_pct", "any_boundary"]].describe(percentiles=[.05,.25,.5,.75,.95]))

{'axis_near_boundary_ratio': 0.199, 'rot_near_boundary_ratio': 0.3265, 'any_boundary_ratio': 0.4605}


,predicted_uplift_pct,delta_pressure_axis_pct,delta_pressure_rotation_pct
count,2000.000000,2000.000000,2000.000000
mean,2.262616,1.148739,2.268927
std,2.234534,5.231691,5.674142
min,-7.502753,-8.000000,-8.000000
5%,0.134146,-8.000000,-8.000000
25%,0.931240,-3.060460,-2.400000
50%,1.735043,1.600000,4.000000
75%,3.041668,5.615113,7.200000
95%,6.401802,8.000000,8.000000
max,22.787921,8.000000,8.000000


## 13. Save artifacts

In [14]:
ARTIFACT_DIR = Path("drilling_advisory_artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)
joblib.dump(rotation_model, ARTIFACT_DIR / "rotation_model.joblib")
joblib.dump(speed_model, ARTIFACT_DIR / "speed_model.joblib")

with open(ARTIFACT_DIR / "surface_ranges_by_energy_type.json", "w", encoding="utf-8") as f:
    json.dump(surface_ranges, f, ensure_ascii=False, indent=2)

feature_config = {
    "data_path": DATA_PATH,
    "rotation_horizon": ROTATION_HORIZON,
    "speed_horizon": SPEED_HORIZON,
    "base_numeric_features": base_numeric_features,
    "categorical_features": categorical_features,
    "speed_numeric_features": speed_numeric_features,
    "speed_extra_features": speed_extra_features,
    "target_rotation": "future_rotation",
    "target_speed": "future_avg_speed",
    "energy_type_column": "rock_energy_type_final",
}
with open(ARTIFACT_DIR / "feature_config.json", "w", encoding="utf-8") as f:
    json.dump(feature_config, f, ensure_ascii=False, indent=2)

optimizer_config = {"grid_size_default": 25, "max_delta_frac_default": 0.08, "use_local_reachable_bounds": True, "use_energy_type_quantile_bounds": True}
with open(ARTIFACT_DIR / "optimizer_config.json", "w", encoding="utf-8") as f:
    json.dump(optimizer_config, f, ensure_ascii=False, indent=2)

metrics_report = {
    "rotation_model": rotation_metrics,
    "speed_model_oracle_rotation": speed_metrics_oracle,
    "speed_model_chained": speed_metrics_chained,
    "baseline_compare": baseline_compare.to_dict(orient="records"),
    "uplift_summary": uplift_summary.to_dict(),
    "uplift_by_energy_type": by_energy.reset_index().to_dict(orient="records"),
    "boundary_report": boundary_report,
}
with open(ARTIFACT_DIR / "training_report.json", "w", encoding="utf-8") as f:
    json.dump(metrics_report, f, ensure_ascii=False, indent=2)
rec_df.to_csv(ARTIFACT_DIR / "offline_recommendations_sample.csv", index=False)

print("Saved artifacts to:", ARTIFACT_DIR.resolve())
for p in sorted(ARTIFACT_DIR.iterdir()):
    print("-", p.name)

Saved artifacts to: C:\Users\qa1ro\OneDrive\Рабочий стол\diploma\OptimalDrilling\notebooks\drilling_advisory_artifacts
- feature_config.json
- offline_recommendations_sample.csv
- optimizer_config.json
- rotation_model.joblib
- speed_model.joblib
- surface_ranges_by_energy_type.json
- training_report.json


## Как читать результат

Хороший признак:
- `speed_model_chained` лучше baseline `current_speed` и `speed_roll_mean_12`.
- median `predicted_uplift_pct` примерно 5–10%, а не огромные 50–100%.
- `any_boundary_ratio` не слишком высокий. Если recommendation почти всегда на границе, optimizer эксплуатирует край сетки.